# Stock Direction Prediction
## Day 2 - Data Cleaning and Feature Engineering

### Objective

The objective of this notebook is to transform historical stock
data into a feature dataset suitable for next-day stock direction
prediction.

The following tasks are performed:

1. Load historical stock data
2. Validate data types
3. Sort data chronologically
4. Remove exact duplicate records
5. Investigate duplicate Ticker-Date combinations
6. Validate OHLC relationships
7. Handle invalid raw observations
8. Analyze missing values
9. Create technical indicators
10. Create lag features
11. Create the next-day UP/DOWN target
12. Remove rows that cannot be used for prediction
13. Save the engineered dataset

No machine learning model is trained in this notebook.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from ta.momentum import RSIIndicator, ROCIndicator, StochasticOscillator
from ta.trend import (
    MACD,
    SMAIndicator,
    EMAIndicator
)
from ta.volatility import (
    BollingerBands,
    AverageTrueRange
)
from ta.volume import (
    OnBalanceVolumeIndicator
)

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [2]:
stock_path = "../data/raw/nifty50_historical_data.csv"

df = pd.read_csv(stock_path)

print("Dataset shape:", df.shape)

Dataset shape: (287310, 25)


In [3]:
print("Columns:")
print(df.columns.tolist())

print("\nTicker exists:", "Ticker" in df.columns)

print("\nShape:", df.shape)

print("\nNumber of stocks:", df["Ticker"].nunique())

Columns:
['Date', 'Ticker', 'Company_Name', 'Sector', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividend', 'Stock_Split', 'Daily_Return', 'Volatility_20D', 'MA_50', 'MA_200', 'Market_Cap', 'PE_Ratio', 'Forward_PE', 'PEG_Ratio', 'Price_to_Book', 'Dividend_Yield', 'EPS', 'Beta', '52Week_High', '52Week_Low']

Ticker exists: True

Shape: (287310, 25)

Number of stocks: 49


In [4]:
df["Date"] = pd.to_datetime(df["Date"])

print(df["Date"].dtype)

datetime64[us, UTC+05:30]


In [5]:
df = df.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

In [6]:
display(
    df[
        ["Ticker", "Date", "Open", "High", "Low", "Close"]
    ].head(20)
)

,Ticker,Date,Open,High,Low,Close
0,ADANIENT.NS,2002-07-01 00:00:00+05:30,-0.012187,-0.012522,-0.011920,-0.012173
1,ADANIENT.NS,2002-07-02 00:00:00+05:30,-0.012385,-0.012426,-0.012118,-0.012269
2,ADANIENT.NS,2002-07-03 00:00:00+05:30,-0.012255,-0.012392,-0.012194,-0.012269
3,ADANIENT.NS,2002-07-04 00:00:00+05:30,-0.012324,-0.012522,-0.012324,-0.012337
4,ADANIENT.NS,2002-07-05 00:00:00+05:30,-0.012406,-0.012406,-0.012262,-0.012310
5,ADANIENT.NS,2002-07-08 00:00:00+05:30,-0.012461,-0.012995,-0.012194,-0.012851
6,ADANIENT.NS,2002-07-09 00:00:00+05:30,-0.012803,-0.012871,-0.012433,-0.012515
7,ADANIENT.NS,2002-07-10 00:00:00+05:30,-0.012597,-0.012597,-0.012392,-0.012502
8,ADANIENT.NS,2002-07-11 00:00:00+05:30,-0.012947,-0.012947,-0.012262,-0.012296
9,ADANIENT.NS,2002-07-12 00:00:00+05:30,-0.012433,-0.012536,-0.012324,-0.012392


In [7]:
exact_duplicates = df.duplicated().sum()
ticker_date_duplicates = df.duplicated(
    subset=["Ticker", "Date"]
).sum()

print("Exact duplicate rows:", exact_duplicates)
print("Duplicate Ticker + Date rows:", ticker_date_duplicates)

Exact duplicate rows: 0
Duplicate Ticker + Date rows: 0


In [8]:
print((df[["Open", "High", "Low", "Close"]] <= 0).sum())
print("Negative volume:", (df["Volume"] < 0).sum())

print(
    "High invalid:",
    ((df["High"] < df["Open"]) | (df["High"] < df["Close"])).sum()
)

print(
    "Low invalid:",
    ((df["Low"] > df["Open"]) | (df["Low"] > df["Close"])).sum()
)

Open     47
High     47
Low      47
Close    47
dtype: int64
Negative volume: 0
High invalid: 123
Low invalid: 119


In [9]:
df[["Open", "High", "Low", "Close", "Volume"]].describe()

,Open,High,Low,Close,Volume
count,287310.000000,287310.000000,287310.000000,287310.000000,2.873100e+05
mean,940.283620,951.740569,927.950491,939.614136,8.064757e+06
std,2457.975005,2486.870766,2426.231206,2455.886319,1.860599e+07
min,-0.012947,-0.012995,-0.012433,-0.012851,0.000000e+00
25%,70.237098,71.436894,68.887683,70.109819,6.296895e+05
50%,236.307412,239.866478,232.499912,236.094742,2.404242e+06
75%,816.242023,826.525145,805.341565,815.629150,8.013830e+06
max,32398.188356,32398.188356,31575.519038,32253.597656,8.552157e+08


In [10]:
df["Return_1D"] = (
    df.groupby("Ticker")["Close"]
    .pct_change()
)

df["Return_5D"] = (
    df.groupby("Ticker")["Close"]
    .pct_change(5)
)

df["Return_10D"] = (
    df.groupby("Ticker")["Close"]
    .pct_change(10)
)

df["Return_20D"] = (
    df.groupby("Ticker")["Close"]
    .pct_change(20)
)

In [11]:
df["SMA_5"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(5).mean())
)

df["SMA_10"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(10).mean())
)

df["SMA_20"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(20).mean())
)

df["SMA_50_new"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(50).mean())
)

In [12]:
df["EMA_12"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x.ewm(span=12, adjust=False).mean())
)

df["EMA_26"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x.ewm(span=26, adjust=False).mean())
)

In [13]:
def calculate_rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1/window,
        min_periods=window,
        adjust=False
    ).mean()

    avg_loss = loss.ewm(
        alpha=1/window,
        min_periods=window,
        adjust=False
    ).mean()

    rs = avg_gain / avg_loss

    return 100 - (100 / (1 + rs))


df["RSI_14"] = (
    df.groupby("Ticker")["Close"]
    .transform(calculate_rsi)
)

In [14]:
df["MACD"] = df["EMA_12"] - df["EMA_26"]

df["MACD_Signal"] = (
    df.groupby("Ticker")["MACD"]
    .transform(lambda x: x.ewm(span=9, adjust=False).mean())
)

df["MACD_Diff"] = df["MACD"] - df["MACD_Signal"]

In [15]:
rolling_mean = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(20).mean())
)

rolling_std = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(20).std())
)

df["BB_Middle"] = rolling_mean
df["BB_High"] = rolling_mean + 2 * rolling_std
df["BB_Low"] = rolling_mean - 2 * rolling_std

df["BB_Width"] = (
    (df["BB_High"] - df["BB_Low"])
    / df["BB_Middle"]
)

In [16]:
df["Previous_Close"] = (
    df.groupby("Ticker")["Close"]
    .shift(1)
)

tr1 = df["High"] - df["Low"]

tr2 = (
    df["High"] - df["Previous_Close"]
).abs()

tr3 = (
    df["Low"] - df["Previous_Close"]
).abs()

df["True_Range"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

df["ATR_14"] = (
    df.groupby("Ticker")["True_Range"]
    .transform(lambda x: x.rolling(14).mean())
)

df.drop(
    columns=["Previous_Close", "True_Range"],
    inplace=True
)

In [17]:
df["Volume_SMA_20"] = (
    df.groupby("Ticker")["Volume"]
    .transform(lambda x: x.rolling(20).mean())
)

df["Volume_Ratio"] = (
    df["Volume"] / df["Volume_SMA_20"]
)

In [18]:
df["Volatility_5D"] = (
    df.groupby("Ticker")["Return_1D"]
    .transform(lambda x: x.rolling(5).std())
)

df["Volatility_10D"] = (
    df.groupby("Ticker")["Return_1D"]
    .transform(lambda x: x.rolling(10).std())
)

df["Volatility_20D_new"] = (
    df.groupby("Ticker")["Return_1D"]
    .transform(lambda x: x.rolling(20).std())
)

In [19]:
df["Price_vs_SMA20"] = (
    df["Close"] / df["SMA_20"] - 1
)

df["Price_vs_SMA50"] = (
    df["Close"] / df["SMA_50_new"] - 1
)

df["EMA12_vs_EMA26"] = (
    df["EMA_12"] / df["EMA_26"] - 1
)

In [20]:
df["High_Low_Range"] = (
    (df["High"] - df["Low"])
    / df["Close"]
)

df["Open_Close_Range"] = (
    (df["Close"] - df["Open"])
    / df["Open"]
)

df["Upper_Shadow"] = (
    df["High"] - df[["Open", "Close"]].max(axis=1)
)

df["Lower_Shadow"] = (
    df[["Open", "Close"]].min(axis=1)
    - df["Low"]
)

In [21]:
df["ROC_10"] = (
    df.groupby("Ticker")["Close"]
    .pct_change(10)
)

df["ROC_20"] = (
    df.groupby("Ticker")["Close"]
    .pct_change(20)
)

In [22]:
df["Close_Lag_1"] = (
    df.groupby("Ticker")["Close"]
    .shift(1)
)

df["Close_Lag_2"] = (
    df.groupby("Ticker")["Close"]
    .shift(2)
)

df["Close_Lag_3"] = (
    df.groupby("Ticker")["Close"]
    .shift(3)
)

df["Return_Lag_1"] = (
    df.groupby("Ticker")["Return_1D"]
    .shift(1)
)

df["Return_Lag_2"] = (
    df.groupby("Ticker")["Return_1D"]
    .shift(2)
)

df["Return_Lag_3"] = (
    df.groupby("Ticker")["Return_1D"]
    .shift(3)
)

In [23]:
df["Next_Close"] = (
    df.groupby("Ticker")["Close"]
    .shift(-1)
)

df["Target"] = (
    df["Next_Close"] > df["Close"]
).astype(int)

In [24]:
print(df["Target"].value_counts())
print(df["Target"].value_counts(normalize=True))

Target
0    143807
1    143503
Name: count, dtype: int64
Target
0    0.500529
1    0.499471
Name: proportion, dtype: float64


In [25]:
feature_columns = [
    "Return_1D",
    "Return_5D",
    "Return_10D",
    "Return_20D",
    "SMA_5",
    "SMA_10",
    "SMA_20",
    "SMA_50_new",
    "EMA_12",
    "EMA_26",
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "MACD_Diff",
    "BB_Middle",
    "BB_High",
    "BB_Low",
    "BB_Width",
    "ATR_14",
    "Volume_SMA_20",
    "Volume_Ratio",
    "Volatility_5D",
    "Volatility_10D",
    "Volatility_20D_new",
    "Price_vs_SMA20",
    "Price_vs_SMA50",
    "EMA12_vs_EMA26",
    "High_Low_Range",
    "Open_Close_Range",
    "Upper_Shadow",
    "Lower_Shadow",
    "ROC_10",
    "ROC_20",
    "Close_Lag_1",
    "Close_Lag_2",
    "Close_Lag_3",
    "Return_Lag_1",
    "Return_Lag_2",
    "Return_Lag_3"
]

required_columns = feature_columns + ["Target"]

df = df.dropna(
    subset=required_columns
).reset_index(drop=True)

print(df.shape)

(282925, 66)


In [26]:
print("Rows:", len(df))
print("Stocks:", df["Ticker"].nunique())
print("Features:", len(feature_columns))

print(
    "\nMissing values in features:"
)

print(
    df[feature_columns]
    .isnull()
    .sum()
    .sum()
)

Rows: 282925
Stocks: 49
Features: 39

Missing values in features:
0


In [27]:
print(df["Target"].value_counts())
print(df["Target"].value_counts(normalize=True))

Target
1    142379
0    140546
Name: count, dtype: int64
Target
1    0.503239
0    0.496761
Name: proportion, dtype: float64


In [28]:
print(feature_columns)

['Return_1D', 'Return_5D', 'Return_10D', 'Return_20D', 'SMA_5', 'SMA_10', 'SMA_20', 'SMA_50_new', 'EMA_12', 'EMA_26', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Diff', 'BB_Middle', 'BB_High', 'BB_Low', 'BB_Width', 'ATR_14', 'Volume_SMA_20', 'Volume_Ratio', 'Volatility_5D', 'Volatility_10D', 'Volatility_20D_new', 'Price_vs_SMA20', 'Price_vs_SMA50', 'EMA12_vs_EMA26', 'High_Low_Range', 'Open_Close_Range', 'Upper_Shadow', 'Lower_Shadow', 'ROC_10', 'ROC_20', 'Close_Lag_1', 'Close_Lag_2', 'Close_Lag_3', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3']


In [29]:
import os

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/stock_technical_features.csv"

df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", df.shape)

Saved: ../data/processed/stock_technical_features.csv
Shape: (282925, 66)


In [30]:
print(df.head())
print(df.tail())
print(df.shape)
print(df.columns.tolist())

                       Date       Ticker            Company_Name  \
0 2002-09-06 00:00:00+05:30  ADANIENT.NS  Adani Enterprises Ltd.   
1 2002-09-09 00:00:00+05:30  ADANIENT.NS  Adani Enterprises Ltd.   
2 2002-09-10 00:00:00+05:30  ADANIENT.NS  Adani Enterprises Ltd.   
3 2002-09-11 00:00:00+05:30  ADANIENT.NS  Adani Enterprises Ltd.   
4 2002-09-12 00:00:00+05:30  ADANIENT.NS  Adani Enterprises Ltd.   

           Sector      Open      High       Low     Close  Volume  Dividend  \
0  Infrastructure  0.050425  0.050425  0.049459  0.050071  510128       0.0   
1  Infrastructure  0.048331  0.049781  0.048331  0.049491  488272       0.0   
2  Infrastructure  0.049491  0.049491  0.049491  0.049491       0       0.0   
3  Infrastructure  0.049910  0.049910  0.048718  0.049298  564188       0.0   
4  Infrastructure  0.048492  0.049459  0.048492  0.049137  504609       0.0   

   Stock_Split  Daily_Return  Volatility_20D     MA_50  MA_200     Market_Cap  \
0          NaN     -0.001927       

In [31]:
df["Momentum_5D"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x - x.shift(5))
)

df["Momentum_10D"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x - x.shift(10))
)

df["Momentum_20D"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x - x.shift(20))
)

df["Momentum_5D_pct"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x / x.shift(5) - 1)
)

df["Momentum_10D_pct"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x / x.shift(10) - 1)
)

df["Momentum_20D_pct"] = (
    df.groupby("Ticker")["Close"]
    .transform(lambda x: x / x.shift(20) - 1)
)

In [32]:
df["Distance_52W_High"] = (
    df["Close"] / df["52Week_High"] - 1
)

df["Distance_52W_Low"] = (
    df["Close"] / df["52Week_Low"] - 1
)

df["52W_Range_Position"] = (
    (df["Close"] - df["52Week_Low"]) /
    (df["52Week_High"] - df["52Week_Low"])
)

In [33]:
df["Body_Size"] = (
    (df["Close"] - df["Open"]).abs()
    / df["Open"]
)

df["Candle_Direction"] = np.where(
    df["Close"] > df["Open"],
    1,
    0
)

df["Range_Position"] = (
    (df["Close"] - df["Low"]) /
    (df["High"] - df["Low"])
)

In [34]:
df["Volume_Change_1D"] = (
    df.groupby("Ticker")["Volume"]
    .pct_change()
)

df["Volume_Change_5D"] = (
    df.groupby("Ticker")["Volume"]
    .pct_change(5)
)

df["Volume_SMA_5"] = (
    df.groupby("Ticker")["Volume"]
    .transform(lambda x: x.rolling(5).mean())
)

df["Volume_SMA_10"] = (
    df.groupby("Ticker")["Volume"]
    .transform(lambda x: x.rolling(10).mean())
)

df["Volume_Ratio_5D"] = (
    df["Volume"] / df["Volume_SMA_5"]
)

df["Volume_Ratio_10D"] = (
    df["Volume"] / df["Volume_SMA_10"]
)

In [35]:
df["Rolling_High_5D"] = (
    df.groupby("Ticker")["High"]
    .transform(lambda x: x.rolling(5).max())
)

df["Rolling_Low_5D"] = (
    df.groupby("Ticker")["Low"]
    .transform(lambda x: x.rolling(5).min())
)

df["Rolling_High_20D"] = (
    df.groupby("Ticker")["High"]
    .transform(lambda x: x.rolling(20).max())
)

df["Rolling_Low_20D"] = (
    df.groupby("Ticker")["Low"]
    .transform(lambda x: x.rolling(20).min())
)

In [36]:
df["Distance_Rolling_High_5D"] = (
    df["Close"] / df["Rolling_High_5D"] - 1
)

df["Distance_Rolling_Low_5D"] = (
    df["Close"] / df["Rolling_Low_5D"] - 1
)

df["Distance_Rolling_High_20D"] = (
    df["Close"] / df["Rolling_High_20D"] - 1
)

df["Distance_Rolling_Low_20D"] = (
    df["Close"] / df["Rolling_Low_20D"] - 1
)

In [37]:
df["RSI_Overbought"] = (
    df["RSI_14"] > 70
).astype(int)

df["RSI_Oversold"] = (
    df["RSI_14"] < 30
).astype(int)

df["RSI_Mid_Bullish"] = (
    (df["RSI_14"] >= 50) &
    (df["RSI_14"] < 70)
).astype(int)

In [38]:
df["SMA20_SMA50_Ratio"] = (
    df["SMA_20"] / df["SMA_50_new"] - 1
)

df["Price_Above_SMA20"] = (
    df["Close"] > df["SMA_20"]
).astype(int)

df["Price_Above_SMA50"] = (
    df["Close"] > df["SMA_50_new"]
).astype(int)

df["EMA12_Above_EMA26"] = (
    df["EMA_12"] > df["EMA_26"]
).astype(int)

In [39]:
df["MACD_Bullish"] = (
    df["MACD"] > df["MACD_Signal"]
).astype(int)

df["MACD_Above_Zero"] = (
    df["MACD"] > 0
).astype(int)

In [40]:
df["BB_Position"] = (
    (df["Close"] - df["BB_Low"]) /
    (df["BB_High"] - df["BB_Low"])
)

In [41]:
df["Volatility_Ratio"] = (
    df["Volatility_5D"] /
    df["Volatility_20D_new"]
)

df["High_Volatility"] = (
    df["Volatility_5D"] >
    df["Volatility_20D_new"]
).astype(int)

In [42]:
df["Trend_Strength_20D"] = (
    df["Close"] / df["Close_Lag_20"]
    if "Close_Lag_20" in df.columns
    else np.nan
)

In [43]:
df["Close_Lag_20"] = (
    df.groupby("Ticker")["Close"]
    .shift(20)
)

df["Trend_Strength_20D"] = (
    df["Close"] / df["Close_Lag_20"] - 1
)

In [44]:
for lag in [5, 10, 20]:
    df[f"Return_Lag_{lag}"] = (
        df.groupby("Ticker")["Return_1D"]
        .shift(lag)
    )

In [45]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Stocks:", df["Ticker"].nunique())

Rows: 282925
Columns: 109
Stocks: 49


In [46]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(
    ascending=False
)

print(missing)

PEG_Ratio                    282925
Stock_Split                  282828
MA_200                         7315
PE_Ratio                       5808
Dividend_Yield                 5808
Beta                           3710
Range_Position                 2232
Momentum_20D                    980
Momentum_20D_pct                980
Return_Lag_20                   980
Trend_Strength_20D              980
Close_Lag_20                    980
Rolling_Low_20D                 931
Rolling_High_20D                931
Distance_Rolling_Low_20D        931
Distance_Rolling_High_20D       931
Volume_Ratio_10D                493
Return_Lag_10                   490
Momentum_10D                    490
Momentum_10D_pct                490
Volume_SMA_10                   441
Volume_Change_5D                428
Volume_Ratio_5D                 278
Momentum_5D                     245
Momentum_5D_pct                 245
Return_Lag_5                    245
Distance_Rolling_High_5D        196
Volume_SMA_5                

In [47]:
print(
    "Infinite values:",
    np.isinf(
        df.select_dtypes(include=np.number)
    ).sum().sum()
)

Infinite values: 4409


In [48]:
df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

,Date,Ticker,Company_Name,Sector,Open,High,Low,Close,Volume,Dividend,Stock_Split,Daily_Return,Volatility_20D,MA_50,MA_200,Market_Cap,PE_Ratio,Forward_PE,PEG_Ratio,Price_to_Book,Dividend_Yield,EPS,Beta,52Week_High,52Week_Low,Return_1D,Return_5D,Return_10D,Return_20D,SMA_5,SMA_10,SMA_20,SMA_50_new,EMA_12,EMA_26,RSI_14,MACD,MACD_Signal,MACD_Diff,BB_Middle,BB_High,BB_Low,BB_Width,ATR_14,Volume_SMA_20,Volume_Ratio,Volatility_5D,Volatility_10D,Volatility_20D_new,Price_vs_SMA20,Price_vs_SMA50,EMA12_vs_EMA26,High_Low_Range,Open_Close_Range,Upper_Shadow,Lower_Shadow,ROC_10,ROC_20,Close_Lag_1,Close_Lag_2,Close_Lag_3,Return_Lag_1,Return_Lag_2,Return_Lag_3,Next_Close,Target,Momentum_5D,Momentum_10D,Momentum_20D,Momentum_5D_pct,Momentum_10D_pct,Momentum_20D_pct,Distance_52W_High,Distance_52W_Low,52W_Range_Position,Body_Size,Candle_Direction,Range_Position,Volume_Change_1D,Volume_Change_5D,Volume_SMA_5,Volume_SMA_10,Volume_Ratio_5D,Volume_Ratio_10D,Rolling_High_5D,Rolling_Low_5D,Rolling_High_20D,Rolling_Low_20D,Distance_Rolling_High_5D,Distance_Rolling_Low_5D,Distance_Rolling_High_20D,Distance_Rolling_Low_20D,RSI_Overbought,RSI_Oversold,RSI_Mid_Bullish,SMA20_SMA50_Ratio,Price_Above_SMA20,Price_Above_SMA50,EMA12_Above_EMA26,MACD_Bullish,MACD_Above_Zero,BB_Position,Volatility_Ratio,High_Volatility,Trend_Strength_20D,Close_Lag_20,Return_Lag_5,Return_Lag_10,Return_Lag_20
0,2002-09-06 00:00:00+05:30,ADANIENT.NS,Adani Enterprises Ltd.,Infrastructure,0.050425,0.050425,0.049459,0.050071,510128,0.0,NaN,-0.001927,1.269080,-0.007684,NaN,2611735429120,31.154974,45.897320,NaN,4.534010,0.06,64.85,0.491,2695.0,1848.0,-0.001927,-5.712252,-5.664168,-5.614143,0.025777,0.007566,-0.001571,-0.007684,0.013269,0.001687,99.121103,0.011582,0.004485,0.007097,-0.001571,0.042982,-0.046123,-56.734209,0.004584,471136.15,1.082761,2.539309,1.794865,1.269080,-32.881089,-7.516113,6.864218,0.019305,-0.007029,0.000000,0.000612,-5.664168,-5.614143,0.050168,0.050103,-0.010715,0.001287,-5.676113,-0.002549,0.049491,0,NaN,NaN,NaN,NaN,NaN,NaN,-0.999981,-0.999973,-2.181759,0.007029,0,0.633339,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,-0.795612,1,1,1,1,1,1.079564,2.000906,1,NaN,NaN,NaN,NaN,NaN
1,2002-09-09 00:00:00+05:30,ADANIENT.NS,Adani Enterprises Ltd.,Infrastructure,0.048331,0.049781,0.048331,0.049491,488272,0.0,NaN,-0.011583,1.268893,-0.006451,NaN,2611735429120,31.154974,45.897320,NaN,4.534010,0.06,64.85,0.491,2695.0,1848.0,-0.011583,-5.607206,-5.624890,-5.540642,0.037824,0.013585,0.001449,-0.006451,0.018842,0.005228,97.976642,0.013614,0.006311,0.007303,0.001449,0.051219,-0.048321,68.697250,0.004701,471070.25,1.036516,2.536788,1.794574,1.268893,33.155903,-8.671952,2.603824,0.029297,0.024000,0.000290,0.000000,-5.624890,-5.540642,0.050071,0.050168,0.050103,-0.001927,0.001287,-5.676113,0.049491,0,NaN,NaN,NaN,NaN,NaN,NaN,-0.999982,-0.999973,-2.181760,0.024000,1,0.800012,-0.042844,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,-1.224616,1,1,1,1,1,0.982638,1.999214,1,NaN,NaN,NaN,NaN,NaN
2,2002-09-10 00:00:00+05:30,ADANIENT.NS,Adani Enterprises Ltd.,Infrastructure,0.049491,0.049491,0.049491,0.049491,0,0.0,NaN,0.000000,1.268937,-0.005216,NaN,2611735429120,31.154974,45.897320,NaN,4.534010,0.06,64.85,0.491,2695.0,1848.0,0.000000,-5.618979,-5.651673,-5.557820,0.049865,0.019598,0.004466,-0.005216,0.023557,0.008507,97.976642,0.015050,0.008059,0.006992,0.004466,0.058251,-0.049318,24.083714,0.004686,451831.65,0.000000,2.537073,1.794775,1.268937,10.080618,-10.488844,1.769142,0.000000,0.000000,0.000000,0.000000,-5.651673,-5.557820,0.049491,0.050071,0.050168,-0.011583,-0.001927,0.001287,0.049298,0,NaN,NaN,NaN,NaN,NaN,NaN,-0.999982,-0.999973,-2.181760,0.000000,0,NaN,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,-1.856346,1,1,1,1,1,0.918566,1.999369,1,NaN,NaN,NaN,NaN,NaN
3,2002-09-11 00:00:00+05:30,ADANIENT.NS,Adani Enterprises Ltd.,Infrastructure,0.049910,0.049910,0.048718,0.049298,564188,0.0,NaN,-0.003906,1.269009,-0.003984,NaN,2611735429120,31

In [49]:
df["Next_Close"] = (
    df.groupby("Ticker")["Close"]
    .shift(-1)
)

df["Target"] = (
    df["Next_Close"] > df["Close"]
).astype(int)

In [50]:
feature_columns = [
    "Return_1D",
    "Return_5D",
    "Return_10D",
    "Return_20D",
    "SMA_5",
    "SMA_10",
    "SMA_20",
    "SMA_50_new",
    "EMA_12",
    "EMA_26",
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "MACD_Diff",
    "BB_Middle",
    "BB_High",
    "BB_Low",
    "BB_Width",
    "ATR_14",
    "Volume_SMA_20",
    "Volume_Ratio",
    "Volatility_5D",
    "Volatility_10D",
    "Volatility_20D_new",
    "Price_vs_SMA20",
    "Price_vs_SMA50",
    "EMA12_vs_EMA26",
    "High_Low_Range",
    "Open_Close_Range",
    "Upper_Shadow",
    "Lower_Shadow",
    "ROC_10",
    "ROC_20",
    "Close_Lag_1",
    "Close_Lag_2",
    "Close_Lag_3",
    "Return_Lag_1",
    "Return_Lag_2",
    "Return_Lag_3",
    "Momentum_5D",
    "Momentum_10D",
    "Momentum_20D",
    "Momentum_5D_pct",
    "Momentum_10D_pct",
    "Momentum_20D_pct",
    "Distance_52W_High",
    "Distance_52W_Low",
    "52W_Range_Position",
    "Body_Size",
    "Candle_Direction",
    "Range_Position",
    "Volume_Change_1D",
    "Volume_Change_5D",
    "Volume_SMA_5",
    "Volume_SMA_10",
    "Volume_Ratio_5D",
    "Volume_Ratio_10D",
    "Rolling_High_5D",
    "Rolling_Low_5D",
    "Rolling_High_20D",
    "Rolling_Low_20D",
    "Distance_Rolling_High_5D",
    "Distance_Rolling_Low_5D",
    "Distance_Rolling_High_20D",
    "Distance_Rolling_Low_20D",
    "RSI_Overbought",
    "RSI_Oversold",
    "RSI_Mid_Bullish",
    "SMA20_SMA50_Ratio",
    "Price_Above_SMA20",
    "Price_Above_SMA50",
    "EMA12_Above_EMA26",
    "MACD_Bullish",
    "MACD_Above_Zero",
    "BB_Position",
    "Volatility_Ratio",
    "High_Volatility",
    "Close_Lag_20",
    "Trend_Strength_20D",
    "Return_Lag_5",
    "Return_Lag_10",
    "Return_Lag_20"
]

In [51]:
required_columns = feature_columns + ["Target"]

df = df.dropna(
    subset=required_columns
).reset_index(drop=True)

print(df.shape)

(275531, 109)


In [52]:
print(
    "Missing feature values:",
    df[feature_columns].isnull().sum().sum()
)

print(
    "Missing target values:",
    df["Target"].isnull().sum()
)

Missing feature values: 0
Missing target values: 0


In [53]:
print(df["Target"].value_counts())

print(
    df["Target"]
    .value_counts(normalize=True)
)

Target
1    138744
0    136787
Name: count, dtype: int64
Target
1    0.503551
0    0.496449
Name: proportion, dtype: float64


In [54]:
print("Start date:", df["Date"].min())
print("End date:", df["Date"].max())

print(
    df.groupby("Ticker")["Date"]
    .agg(["min", "max"])
    .head(10)
)

Start date: 1999-04-08 00:00:00+05:30
End date: 2026-01-30 00:00:00+05:30
                                    min                       max
Ticker                                                           
ADANIENT.NS   2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
ADANIPORTS.NS 2008-03-05 00:00:00+05:30 2026-01-30 00:00:00+05:30
APOLLOHOSP.NS 2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
ASIANPAINT.NS 2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
AXISBANK.NS   1999-04-08 00:00:00+05:30 2026-01-30 00:00:00+05:30
BAJAJ-AUTO.NS 2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
BAJAJFINSV.NS 2002-11-15 00:00:00+05:30 2026-01-30 00:00:00+05:30
BAJFINANCE.NS 2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
BHARTIARTL.NS 2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
BPCL.NS       1999-04-08 00:00:00+05:30 2026-01-30 00:00:00+05:30


In [55]:
corr = (
    df[feature_columns + ["Target"]]
    .corr()["Target"]
    .sort_values()
)

print(corr.head(10))
print(corr.tail(10))

RSI_Overbought             -0.012621
Return_5D                  -0.012402
Momentum_5D_pct            -0.012402
Price_vs_SMA20             -0.011444
RSI_14                     -0.011137
Price_vs_SMA50             -0.009922
BB_Position                -0.008649
Distance_Rolling_Low_20D   -0.008453
ROC_10                     -0.008412
Return_10D                 -0.008412
Name: Target, dtype: float64
Distance_52W_High    0.003283
RSI_Mid_Bullish      0.003784
Volume_SMA_5         0.004848
Volume_Ratio         0.004941
Volume_SMA_10        0.005050
Volume_SMA_20        0.005313
Volume_Ratio_10D     0.006259
RSI_Oversold         0.006532
Volume_Ratio_5D      0.008077
Target               1.000000
Name: Target, dtype: float64


In [56]:
df[feature_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Return_1D,275531.0,0.001052,0.029146,-0.926483,-0.010032,0.000292,0.011227,7.440012
Return_5D,275531.0,0.005302,0.063849,-0.925923,-0.021302,0.003370,0.029778,8.472941
Return_10D,275531.0,0.010827,0.100438,-0.990356,-0.028044,0.007714,0.046081,12.494975
Return_20D,275531.0,0.021888,0.135212,-0.990744,-0.036736,0.015775,0.073320,13.015819
SMA_5,275531.0,968.532130,2493.967529,0.039039,76.604964,247.927283,851.846014,31648.312891
...,...,...,...,...,...,...,...,...
Close_Lag_20,275531.0,958.266554,2476.990963,0.038697,75.541149,244.749191,841.239777,32253.597656
Trend_Strength_20D,275531.0,0.021888,0.135212,-0.990744,-0.036736,0.015775,0.073320,13.015819
Return_Lag_5,275531.0,0.001191,0.054390,-0.990858,-0.010020,0.000313,0.011267,12.241736
Return_Lag_10,275531.0,0.001990,0.219754,-0.990858,-0.009903,0.000209,0.011183,108.493070


In [57]:
import os

os.makedirs(
    "../data/processed",
    exist_ok=True
)

output_path = (
    "../data/processed/"
    "stock_technical_features.csv"
)

df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", df.shape)

Saved: ../data/processed/stock_technical_features.csv
Shape: (275531, 109)


In [58]:
feature_df = pd.DataFrame({
    "Feature": feature_columns
})

feature_df.to_csv(
    "../data/processed/feature_list.csv",
    index=False
)

print("Feature list saved")

Feature list saved


In [59]:
print("Final shape:", df.shape)
print("Number of stocks:", df["Ticker"].nunique())
print("Number of features:", len(feature_columns))

print(
    "Missing values:",
    df[feature_columns].isnull().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(
        df[feature_columns]
        .select_dtypes(include=np.number)
    ).sum().sum()
)

Final shape: (275531, 109)
Number of stocks: 49
Number of features: 82
Missing values: 0
Infinite values: 0


In [60]:
df[
    [
        "Date",
        "Ticker",
        "Close",
        "Return_1D",
        "RSI_14",
        "MACD",
        "BB_Position",
        "Volume_Ratio",
        "Momentum_5D_pct",
        "Trend_Strength_20D",
        "Target"
    ]
].head(20)

,Date,Ticker,Close,Return_1D,RSI_14,MACD,BB_Position,Volume_Ratio,Momentum_5D_pct,Trend_Strength_20D,Target
0,2002-10-04 00:00:00+05:30,ADANIENT.NS,0.043143,-0.003720,72.164685,0.007472,0.039700,1.059353,-0.070139,-0.138353,1
1,2002-10-07 00:00:00+05:30,ADANIENT.NS,0.044110,0.022405,73.773711,0.006907,0.182700,1.061017,-0.029766,-0.108724,1
2,2002-10-08 00:00:00+05:30,ADANIENT.NS,0.046688,0.058436,77.507548,0.006591,0.476059,1.050535,0.065441,-0.056641,1
3,2002-10-10 00:00:00+05:30,ADANIENT.NS,0.046559,-0.028898,73.040337,0.006011,0.482656,1.016705,0.075149,-0.052459,1
4,2002-10-11 00:00:00+05:30,ADANIENT.NS,0.047493,0.020069,74.455543,0.005737,0.609552,1.120916,0.100822,-0.039739,0
5,2002-10-14 00:00:00+05:30,ADANIENT.NS,0.047300,-0.004071,73.594647,0.005442,0.600070,0.995717,0.072315,-0.035480,0
6,2002-10-17 00:00:00+05:30,ADANIENT.NS,0.046913,-0.002739,71.624862,0.004531,0.595061,1.071960,0.007613,-0.035761,0
7,2002-10-18 00:00:00+05:30,ADANIENT.NS,0.046656,-0.005494,70.208195,0.004218,0.574276,1.052310,-0.017639,-0.037873,1
8,2002-10-21 00:00:00+05:30,ADANIENT.NS,0.046688,0.000690,70.287233,0.003927,0.600840,1.058185,-0.012943,-0.041033,0
9,2002-10-23 00:00:00+05:30,ADANIENT.NS,0.046688,0.001381,70.070961,0.003395,0.609896,1.099534,-0.007534,-0.002754,0


In [61]:
leakage_words = [
    "Next",
    "Target"
]

print([
    col for col in feature_columns
    if any(word in col for word in leakage_words)
])

[]


In [62]:
check_order = (
    df.groupby("Ticker")["Date"]
    .apply(lambda x: x.is_monotonic_increasing)
)

print(check_order.all())
print(check_order.value_counts())

True
Date
True    49
Name: count, dtype: int64


In [63]:
df.to_csv(
    "../data/processed/stock_technical_features.csv",
    index=False
)

print("Day 2 dataset successfully saved.")

Day 2 dataset successfully saved.


In [64]:
print(df.shape)
print(len(feature_columns))

(275531, 109)
82


In [65]:
print(
    df["Target"].value_counts(normalize=True)
)

print(
    df.groupby("Ticker")["Date"]
    .agg(["min", "max"])
)

Target
1    0.503551
0    0.496449
Name: proportion, dtype: float64
                                    min                       max
Ticker                                                           
ADANIENT.NS   2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
ADANIPORTS.NS 2008-03-05 00:00:00+05:30 2026-01-30 00:00:00+05:30
APOLLOHOSP.NS 2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
ASIANPAINT.NS 2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
AXISBANK.NS   1999-04-08 00:00:00+05:30 2026-01-30 00:00:00+05:30
BAJAJ-AUTO.NS 2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
BAJAJFINSV.NS 2002-11-15 00:00:00+05:30 2026-01-30 00:00:00+05:30
BAJFINANCE.NS 2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
BHARTIARTL.NS 2002-10-04 00:00:00+05:30 2026-01-30 00:00:00+05:30
BPCL.NS       1999-04-08 00:00:00+05:30 2026-01-30 00:00:00+05:30
BRITANNIA.NS  1999-04-08 00:00:00+05:30 2026-01-30 00:00:00+05:30
CIPLA.NS      1999-04-08 00:00:00+05:30 2026-01-30 00:00:00+05:30
COALINDI

In [66]:
check_order = (
    df.groupby("Ticker")["Date"]
    .apply(lambda x: x.is_monotonic_increasing)
)

print("All stocks chronological:", check_order.all())
print(check_order.value_counts())

All stocks chronological: True
Date
True    49
Name: count, dtype: int64


In [67]:
duplicate_dates = (
    df.groupby(["Ticker", "Date"])
    .size()
    .reset_index(name="count")
)

print(
    duplicate_dates[
        duplicate_dates["count"] > 1
    ]
)

Empty DataFrame
Columns: [Ticker, Date, count]
Index: []


In [68]:
target_by_stock = (
    df.groupby("Ticker")["Target"]
    .agg(["count", "mean"])
    .sort_values("mean")
)

print(target_by_stock)

               count      mean
Ticker                        
CIPLA.NS        6501  0.488079
AXISBANK.NS     6493  0.489604
EICHERMOT.NS    6474  0.489960
HINDUNILVR.NS   6499  0.491768
HDFCLIFE.NS     1952  0.491803
APOLLOHOSP.NS   5697  0.492013
BRITANNIA.NS    6467  0.493737
TATACONSUM.NS   6504  0.494465
BPCL.NS         6502  0.495694
POWERGRID.NS    4428  0.496838
BHARTIARTL.NS   5697  0.498859
NTPC.NS         5149  0.498932
MARUTI.NS       5450  0.500734
INDUSINDBK.NS   5695  0.500966
ONGC.NS         6500  0.501385
BAJAJFINSV.NS   5561  0.501888
LT.NS           5665  0.502383
TITAN.NS        6499  0.502385
ITC.NS          6504  0.502460
ADANIPORTS.NS   4384  0.502509
HINDALCO.NS     6499  0.503000
TATASTEEL.NS    6499  0.503308
SUNPHARMA.NS    6498  0.503847
HDFCBANK.NS     6498  0.504309
SHREECEM.NS     5926  0.504556
ADANIENT.NS     5681  0.504841
ICICIBANK.NS    5697  0.505003
JSWSTEEL.NS     5442  0.505145
COALINDIA.NS    3678  0.505438
HEROMOTOCO.NS   5697  0.505529
BAJFINAN

In [69]:
df[
    [
        "Return_1D",
        "Return_5D",
        "RSI_14",
        "MACD",
        "ATR_14",
        "BB_Position",
        "Volume_Ratio",
        "Momentum_5D_pct",
        "Trend_Strength_20D"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
Return_1D,275531.0,0.001052,0.029146,-0.926483,-0.010032,0.000292,0.011227,7.440012
Return_5D,275531.0,0.005302,0.063849,-0.925923,-0.021302,0.003370,0.029778,8.472941
RSI_14,275531.0,52.828109,12.278599,1.395207,44.318271,52.863445,61.457269,98.837513
MACD,275531.0,3.964472,48.146732,-1616.712465,-1.364846,0.515115,5.166757,1144.277755
ATR_14,275531.0,25.335427,67.205917,0.000525,2.553381,7.387892,22.035349,1973.676083
BB_Position,275531.0,0.550420,0.324251,-0.544184,0.290256,0.576825,0.807161,1.561716
Volume_Ratio,275531.0,1.026261,0.749958,0.000000,0.619186,0.860988,1.206481,19.211980
Momentum_5D_pct,275531.0,0.005302,0.063849,-0.925923,-0.021302,0.003370,0.029778,8.472941
Trend_Strength_20D,275531.0,0.021888,0.135212,-0.990744,-0.036736,0.015775,0.073320,13.015819


In [70]:
print("RSI minimum:", df["RSI_14"].min())
print("RSI maximum:", df["RSI_14"].max())

print(
    "RSI outside 0-100:",
    ((df["RSI_14"] < 0) | (df["RSI_14"] > 100)).sum()
)

RSI minimum: 1.3952069261187034
RSI maximum: 98.83751325678749
RSI outside 0-100: 0


In [71]:
print("BB Position min:", df["BB_Position"].min())
print("BB Position max:", df["BB_Position"].max())

print(
    "BB Position missing:",
    df["BB_Position"].isnull().sum()
)

BB Position min: -0.5441838118694439
BB Position max: 1.561715728463234
BB Position missing: 0


In [72]:
print(
    df["Return_1D"]
    .abs()
    .sort_values(ascending=False)
    .head(20)
)

206       7.440012
269150    3.999308
64664     3.861169
392       1.094372
79797     1.045164
167288    0.950958
34814     0.926483
64665     0.795261
269147    0.783991
263778    0.756378
34769     0.637851
167287    0.505669
79800     0.499784
166736    0.451142
57143     0.450176
140568    0.446731
3052      0.387493
156241    0.337108
119886    0.315594
258412    0.291832
Name: Return_1D, dtype: float64


In [73]:
extreme_returns = (
    df.loc[
        df["Return_1D"].abs() > 0.20,
        [
            "Date",
            "Ticker",
            "Open",
            "High",
            "Low",
            "Close",
            "Return_1D",
            "Stock_Split"
        ]
    ]
    .sort_values("Return_1D")
)

print(extreme_returns.head(20))

                            Date         Ticker         Open         High  \
34814  2008-05-26 00:00:00+05:30  BAJAJFINSV.NS    57.702561    59.529809   
64665  2004-05-12 00:00:00+05:30       CIPLA.NS    86.675254    90.436312   
269147 1999-09-22 00:00:00+05:30       WIPRO.NS     3.543132     3.543274   
263778 2004-08-24 00:00:00+05:30  ULTRACEMCO.NS   277.012582   308.800911   
34769  2008-03-14 00:00:00+05:30  BAJAJFINSV.NS   739.540824  1037.608569   
167287 2006-09-27 00:00:00+05:30          LT.NS   115.200303   115.577310   
79800  2001-10-15 00:00:00+05:30     DRREDDY.NS    80.011378    80.699650   
166736 2004-05-19 00:00:00+05:30          LT.NS    42.997875    42.997875   
3052   2015-06-03 00:00:00+05:30    ADANIENT.NS   294.325824   294.325824   
4947   2023-02-01 00:00:00+05:30    ADANIENT.NS  2990.897042  3006.625465   
57141  1999-09-22 00:00:00+05:30   BRITANNIA.NS    51.611484    51.644360   
26506  2020-03-23 00:00:00+05:30    AXISBANK.NS   383.747696   390.370039   

In [74]:
split_data = df[
    df["Stock_Split"].notna() &
    (df["Stock_Split"] != 0)
][
    [
        "Date",
        "Ticker",
        "Close",
        "Stock_Split"
    ]
]

print("Stock split records:", len(split_data))
print(split_data.head(20))

Stock split records: 94
                           Date         Ticker        Close  Stock_Split
1709  2009-12-10 00:00:00+05:30    ADANIENT.NS    54.488613          2.0
6294  2010-09-23 00:00:00+05:30  ADANIPORTS.NS   152.392731          5.0
11971 2010-09-02 00:00:00+05:30  APOLLOHOSP.NS   386.152039          2.0
18387 2013-07-30 00:00:00+05:30  ASIANPAINT.NS   461.131134         10.0
25122 2014-07-28 00:00:00+05:30    AXISBANK.NS   382.707031          5.0
29796 2010-09-08 00:00:00+05:30  BAJAJ-AUTO.NS  1005.158020          2.0
38310 2022-09-13 00:00:00+05:30  BAJAJFINSV.NS  1781.817139          5.0
42517 2016-09-08 00:00:00+05:30  BAJFINANCE.NS   112.957428          5.0
44678 2025-06-16 00:00:00+05:30  BAJFINANCE.NS   938.000000          2.0
46467 2009-07-24 00:00:00+05:30  BHARTIARTL.NS   345.442719          2.0
50926 2000-12-20 00:00:00+05:30        BPCL.NS     3.679740          2.0
53703 2012-07-13 00:00:00+05:30        BPCL.NS    32.149738          2.0
54677 2016-07-13 00:00:00+0

In [75]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Stocks:", df["Ticker"].nunique())

print(
    "Missing feature values:",
    df[feature_columns].isnull().sum().sum()
)

print(
    "Missing targets:",
    df["Target"].isnull().sum()
)

print(
    "Infinite feature values:",
    np.isinf(
        df[feature_columns]
        .select_dtypes(include=np.number)
    ).sum().sum()
)

Rows: 275531
Columns: 109
Stocks: 49
Missing feature values: 0
Missing targets: 0
Infinite feature values: 0


In [76]:
future_columns = [
    col for col in feature_columns
    if "Next" in col
    or "Future" in col
]

print("Potential future-leakage features:")
print(future_columns)

Potential future-leakage features:
[]


In [77]:
print("Total features:", len(feature_columns))

for i, feature in enumerate(feature_columns, 1):
    print(i, feature)

Total features: 82
1 Return_1D
2 Return_5D
3 Return_10D
4 Return_20D
5 SMA_5
6 SMA_10
7 SMA_20
8 SMA_50_new
9 EMA_12
10 EMA_26
11 RSI_14
12 MACD
13 MACD_Signal
14 MACD_Diff
15 BB_Middle
16 BB_High
17 BB_Low
18 BB_Width
19 ATR_14
20 Volume_SMA_20
21 Volume_Ratio
22 Volatility_5D
23 Volatility_10D
24 Volatility_20D_new
25 Price_vs_SMA20
26 Price_vs_SMA50
27 EMA12_vs_EMA26
28 High_Low_Range
29 Open_Close_Range
30 Upper_Shadow
31 Lower_Shadow
32 ROC_10
33 ROC_20
34 Close_Lag_1
35 Close_Lag_2
36 Close_Lag_3
37 Return_Lag_1
38 Return_Lag_2
39 Return_Lag_3
40 Momentum_5D
41 Momentum_10D
42 Momentum_20D
43 Momentum_5D_pct
44 Momentum_10D_pct
45 Momentum_20D_pct
46 Distance_52W_High
47 Distance_52W_Low
48 52W_Range_Position
49 Body_Size
50 Candle_Direction
51 Range_Position
52 Volume_Change_1D
53 Volume_Change_5D
54 Volume_SMA_5
55 Volume_SMA_10
56 Volume_Ratio_5D
57 Volume_Ratio_10D
58 Rolling_High_5D
59 Rolling_Low_5D
60 Rolling_High_20D
61 Rolling_Low_20D
62 Distance_Rolling_High_5D
63 Dista

In [78]:
model_df = df[
    ["Date", "Ticker"] + feature_columns + ["Target"]
].copy()

print(model_df.shape)
print(model_df.head())

(275531, 85)
                       Date       Ticker  Return_1D  Return_5D  Return_10D  \
0 2002-10-04 00:00:00+05:30  ADANIENT.NS  -0.003720  -0.070139   -0.110299   
1 2002-10-07 00:00:00+05:30  ADANIENT.NS   0.022405  -0.029766   -0.093978   
2 2002-10-08 00:00:00+05:30  ADANIENT.NS   0.058436   0.065441   -0.014286   
3 2002-10-10 00:00:00+05:30  ADANIENT.NS  -0.028898   0.075149    0.000000   
4 2002-10-11 00:00:00+05:30  ADANIENT.NS   0.020069   0.100822    0.023611   

   Return_20D     SMA_5    SMA_10    SMA_20  SMA_50_new    EMA_12    EMA_26  \
0   -0.138353  0.043910  0.045538  0.047311    0.016115  0.044258  0.036786   
1   -0.108724  0.043640  0.045080  0.047042    0.017219  0.044235  0.037328   
2   -0.056641  0.044213  0.045012  0.046902    0.018373  0.044612  0.038021   
3   -0.052459  0.045689  0.045125  0.046705    0.020701  0.045346  0.039334   
4   -0.039739  0.046559  0.045235  0.046607    0.021869  0.045676  0.039939   

      RSI_14      MACD  MACD_Signal  MACD_D

In [79]:
print("Rows:", len(model_df))
print("Columns:", len(model_df.columns))
print("Features:", len(feature_columns))

print(
    "Missing values:",
    model_df[feature_columns + ["Target"]]
    .isnull()
    .sum()
    .sum()
)

Rows: 275531
Columns: 85
Features: 82
Missing values: 0


In [80]:
model_path = (
    "../data/processed/"
    "stock_model_dataset.csv"
)

model_df.to_csv(
    model_path,
    index=False
)

print("Saved:", model_path)

Saved: ../data/processed/stock_model_dataset.csv


In [81]:
feature_df = pd.DataFrame({
    "Feature": feature_columns
})

feature_df.to_csv(
    "../data/processed/feature_list.csv",
    index=False
)

print("Saved feature list")

Saved feature list


In [82]:
print("========== DAY 2 SUMMARY ==========")
print("Rows:", len(model_df))
print("Features:", len(feature_columns))
print("Stocks:", model_df["Ticker"].nunique())
print("Start:", model_df["Date"].min())
print("End:", model_df["Date"].max())
print("UP:", (model_df["Target"] == 1).sum())
print("DOWN:", (model_df["Target"] == 0).sum())
print(
    "Missing:",
    model_df[feature_columns + ["Target"]]
    .isnull()
    .sum()
    .sum()
)

========== DAY 2 SUMMARY ==========
Rows: 275531
Features: 82
Stocks: 49
Start: 1999-04-08 00:00:00+05:30
End: 2026-01-30 00:00:00+05:30
UP: 138744
DOWN: 136787
Missing: 0


In [83]:
stock_counts = model_df.groupby("Ticker").size().sort_values()

print(stock_counts)
print()
print("Minimum observations:", stock_counts.min())
print("Maximum observations:", stock_counts.max())

Ticker
HDFCLIFE.NS      1952
SBILIFE.NS       1984
LTIM.NS          2272
COALINDIA.NS     3678
NESTLEIND.NS     4023
ADANIPORTS.NS    4384
POWERGRID.NS     4428
TECHM.NS         4703
NTPC.NS          5149
TCS.NS           5405
JSWSTEEL.NS      5442
MARUTI.NS        5450
DIVISLAB.NS      5540
BAJAJFINSV.NS    5561
HCLTECH.NS       5584
BAJAJ-AUTO.NS    5628
ULTRACEMCO.NS    5654
LT.NS            5665
ADANIENT.NS      5681
BAJFINANCE.NS    5692
INDUSINDBK.NS    5695
GRASIM.NS        5696
BHARTIARTL.NS    5697
APOLLOHOSP.NS    5697
ICICIBANK.NS     5697
HEROMOTOCO.NS    5697
ASIANPAINT.NS    5697
KOTAKBANK.NS     5921
SHREECEM.NS      5926
BRITANNIA.NS     6467
EICHERMOT.NS     6474
AXISBANK.NS      6493
WIPRO.NS         6493
INFY.NS          6494
SUNPHARMA.NS     6498
HDFCBANK.NS      6498
TATASTEEL.NS     6499
HINDALCO.NS      6499
HINDUNILVR.NS    6499
TITAN.NS         6499
M&M.NS           6500
ONGC.NS          6500
CIPLA.NS         6501
BPCL.NS          6502
DRREDDY.NS       6502
SBI

In [84]:
target_balance = (
    model_df.groupby("Ticker")["Target"]
    .mean()
    .sort_values()
)

print(target_balance)

Ticker
CIPLA.NS         0.488079
AXISBANK.NS      0.489604
EICHERMOT.NS     0.489960
HINDUNILVR.NS    0.491768
HDFCLIFE.NS      0.491803
APOLLOHOSP.NS    0.492013
BRITANNIA.NS     0.493737
TATACONSUM.NS    0.494465
BPCL.NS          0.495694
POWERGRID.NS     0.496838
BHARTIARTL.NS    0.498859
NTPC.NS          0.498932
MARUTI.NS        0.500734
INDUSINDBK.NS    0.500966
ONGC.NS          0.501385
BAJAJFINSV.NS    0.501888
LT.NS            0.502383
TITAN.NS         0.502385
ITC.NS           0.502460
ADANIPORTS.NS    0.502509
HINDALCO.NS      0.503000
TATASTEEL.NS     0.503308
SUNPHARMA.NS     0.503847
HDFCBANK.NS      0.504309
SHREECEM.NS      0.504556
ADANIENT.NS      0.504841
ICICIBANK.NS     0.505003
JSWSTEEL.NS      0.505145
COALINDIA.NS     0.505438
HEROMOTOCO.NS    0.505529
BAJFINANCE.NS    0.507554
KOTAKBANK.NS     0.507853
GRASIM.NS        0.507900
WIPRO.NS         0.507932
SBIN.NS          0.509457
TCS.NS           0.509528
NESTLEIND.NS     0.509570
RELIANCE.NS      0.509686
DRRED

In [85]:
print(
    "Exact duplicates:",
    model_df.duplicated().sum()
)

print(
    "Ticker-Date duplicates:",
    model_df.duplicated(
        subset=["Ticker", "Date"]
    ).sum()
)

Exact duplicates: 0
Ticker-Date duplicates: 0


In [86]:
print(
    model_df.dtypes.value_counts()
)

print()

print(
    model_df.dtypes
)

float64                      72
int64                        11
datetime64[us, UTC+05:30]     1
str                           1
Name: count, dtype: int64

Date                  datetime64[us, UTC+05:30]
Ticker                                      str
Return_1D                               float64
Return_5D                               float64
Return_10D                              float64
                                ...            
Trend_Strength_20D                      float64
Return_Lag_5                            float64
Return_Lag_10                           float64
Return_Lag_20                           float64
Target                                    int64
Length: 85, dtype: object


In [87]:
numeric_features = model_df[
    feature_columns
].select_dtypes(
    include=np.number
).columns.tolist()

print("Expected features:", len(feature_columns))
print("Numeric features:", len(numeric_features))

Expected features: 82
Numeric features: 82


In [88]:
print("Target in features:", "Target" in feature_columns)
print("Next_Close in features:", "Next_Close" in feature_columns)

Target in features: False
Next_Close in features: False


In [89]:
print(
    "Memory usage:",
    round(
        model_df.memory_usage(deep=True).sum() / 1024**2,
        2
    ),
    "MB"
)

Memory usage: 192.17 MB


In [90]:
model_df.to_csv(
    "../data/processed/stock_model_dataset.csv",
    index=False
)

pd.DataFrame({
    "Feature": feature_columns
}).to_csv(
    "../data/processed/feature_list.csv",
    index=False
)

print("All Day 2 files saved successfully.")

All Day 2 files saved successfully.


In [91]:
sample = model_df.sample(
    n=min(1000, len(model_df)),
    random_state=42
)

sample.to_csv(
    "../data/processed/model_sample.csv",
    index=False
)

print(sample.shape)

(1000, 85)


In [92]:
print("========== DAY 2 COMPLETE ==========")
print()
print("Dataset shape:", model_df.shape)
print("Number of stocks:", model_df["Ticker"].nunique())
print("Number of features:", len(feature_columns))
print("Start date:", model_df["Date"].min())
print("End date:", model_df["Date"].max())
print()
print("UP:", (model_df["Target"] == 1).sum())
print("DOWN:", (model_df["Target"] == 0).sum())
print()
print("Missing values:", model_df[feature_columns].isnull().sum().sum())
print("Duplicate rows:", model_df.duplicated().sum())
print(
    "Duplicate Ticker-Date:",
    model_df.duplicated(
        subset=["Ticker", "Date"]
    ).sum()
)
print()
print("Target in features:", "Target" in feature_columns)
print("Next_Close in features:", "Next_Close" in feature_columns)

========== DAY 2 COMPLETE ==========

Dataset shape: (275531, 85)
Number of stocks: 49
Number of features: 82
Start date: 1999-04-08 00:00:00+05:30
End date: 2026-01-30 00:00:00+05:30

UP: 138744
DOWN: 136787

Missing values: 0
Duplicate rows: 0
Duplicate Ticker-Date: 0

Target in features: False
Next_Close in features: False
